# Buscar en un mapa: BFS, DFS, coste uniforme y A\*

**Facsímil 2 · Inteligencia clásica** — capítulos 1 a 3
(búsqueda como espacio de estados, algoritmos ciegos, y A\* con heurísticas).

Antes de los LLM, *esto* era la inteligencia artificial: representar un problema como un espacio de
estados y buscar dentro de él. Y sigue estando por debajo de un GPS, de un planificador de repartos o
de un agente que decide pasos hacia un objetivo. En este cuaderno sueltas un repartidor en una
ciudad-cuadrícula con calles cortadas y comparas varios algoritmos de búsqueda. No solo si encuentran
la ruta: **cuánto tienen que mirar** para encontrarla. Esa diferencia —entre buscar a ciegas y buscar
con criterio— es una de las ideas más rentables de toda la informática.

### Qué vas a aprender
- Qué es un **espacio de estados** y por qué casi cualquier problema puede plantearse como una
  búsqueda en él.
- Cómo **BFS, DFS, coste uniforme (UCS) y A\*** exploran ese espacio de formas distintas, y qué
  **garantiza** cada uno (¿encuentra solución? ¿la óptima? ¿a qué coste?).
- Qué es una **heurística**, por qué una buena convierte una búsqueda imposible en una instantánea, y
  qué significa que sea **admisible** (con Manhattan y con la distancia euclídea).
- La **búsqueda voraz** (solo heurística) y el **A\* ponderado**: la palanca entre **velocidad y
  optimalidad**, y qué pasa cuando dejas de ser admisible. Lo verás con números, no de palabra.
- Por qué en un **laberinto** la DFS hace el ridículo, y por qué con **terreno de coste variable** (barro)
  la BFS deja de dar la ruta más barata mientras UCS y A\* la siguen dando.

### Cuánto cuesta
Unos 18 minutos. CPU, solo librería estándar y Matplotlib.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Una explanada grande con un edificio en medio que hay que rodear. En campo
# abierto se ve clarisimo quien busca a ciegas y quien busca con criterio.
GRID = np.zeros((20, 32), dtype=int)   # 0 = calle, 1 = obstaculo
GRID[5:15, 11:21] = 1                   # el edificio central
INICIO, META = (18, 2), (1, 29)
print("Mapa de", GRID.shape, "casillas. Inicio:", INICIO, " Meta:", META)


## 1. Buscar = explorar un espacio de estados

Casi cualquier problema de decisión puede plantearse así: hay **estados** (situaciones), un estado
**inicial**, uno o varios estados **meta**, y **acciones** que llevan de un estado a otro con un
cierto coste. Resolver el problema es encontrar una secuencia de acciones del inicial a la meta. En
nuestro mapa, cada casilla es un estado; las acciones son moverse a una casilla vecina (arriba, abajo,
izquierda, derecha) si no es un muro.

Cambia el mapa por otra cosa y la idea no se mueve. Un GPS busca en el espacio de cruces y tramos de
calle; un planificador de repartos, en el espacio de órdenes de visita; un agente que encadena pasos,
en el espacio de situaciones del mundo. Siempre lo mismo: estados, acciones, coste, y la pregunta de
cómo llegar.

La pregunta no es solo «¿hay camino?», sino «¿cuántos estados tienes que **expandir** (mirar) para
dar con él?». En problemas reales el espacio de estados es astronómico (un ajedrez tiene más estados
que átomos hay en el universo observable), así que *cuánto* exploras lo es todo.


In [ ]:
def vecinos(p):
    f, c = p
    for df, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nf, nc = f+df, c+dc
        if 0 <= nf < GRID.shape[0] and 0 <= nc < GRID.shape[1] and GRID[nf, nc] == 0:
            yield (nf, nc)

def reconstruir(padre, fin):
    camino = [fin]
    while camino[-1] in padre:
        camino.append(padre[camino[-1]])
    return camino[::-1]


## 2. Cuatro formas de explorar, y qué garantiza cada una

La diferencia entre los algoritmos está en **el orden** en que sacan estados de la frontera (los
candidatos por explorar). Y ese orden depende de una estructura de datos:

| Algoritmo | Estructura | ¿Encuentra solución? | ¿La más corta? | Idea |
|---|---|---|---|---|
| **BFS** (anchura) | cola (FIFO) | sí | **sí** (en pasos) | explora por capas, como una onda en el agua |
| **DFS** (profundidad) | pila (LIFO) | sí | **no** | tira por un camino hasta el fondo |
| **Coste uniforme (UCS)** | cola de prioridad por coste | sí | **sí** (en coste) | como BFS pero ordenando por coste acumulado |
| **A\*** | cola de prioridad por coste + heurística | sí | **sí** (si la heurística es admisible) | va *hacia* la meta, no en todas direcciones |

La imagen de la **onda** ayuda con BFS: tira una piedra en el agua en el punto de inicio y la onda
crece anillo a anillo; el primer anillo que toca la meta marca el camino más corto en pasos. UCS es
esa misma onda, pero avanzando más despacio por donde el terreno cuesta más.

La clave de A\* es la **heurística** $h(n)$: una estimación de lo que falta desde $n$ hasta la meta.
A\* ordena la frontera por $f(n) = g(n) + h(n)$, donde $g$ es el coste ya recorrido. Usaremos la
**distancia de Manhattan** (cuántas casillas en horizontal más en vertical), que en una cuadrícula
nunca sobreestima: es *admisible*, y por eso A\* sigue dando el camino óptimo.


In [ ]:
def bfs(inicio, meta):
    frontera = deque([inicio]); padre = {}; vistos = {inicio}; expandidos = 0
    while frontera:
        n = frontera.popleft(); expandidos += 1
        if n == meta: return reconstruir(padre, meta), expandidos, vistos
        for v in vecinos(n):
            if v not in vistos:
                vistos.add(v); padre[v] = n; frontera.append(v)
    return None, expandidos, vistos

def dfs(inicio, meta):
    pila = [inicio]; padre = {}; vistos = {inicio}; expandidos = 0
    while pila:
        n = pila.pop(); expandidos += 1
        if n == meta: return reconstruir(padre, meta), expandidos, vistos
        for v in vecinos(n):
            if v not in vistos:
                vistos.add(v); padre[v] = n; pila.append(v)
    return None, expandidos, vistos

def ucs(inicio, meta):
    pq = [(0, inicio)]; padre = {}; coste = {inicio: 0}; expandidos = 0; vistos = set()
    while pq:
        g, n = heapq.heappop(pq)
        if n in vistos: continue
        vistos.add(n); expandidos += 1
        if n == meta: return reconstruir(padre, meta), expandidos, vistos
        for v in vecinos(n):
            ng = g + 1
            if ng < coste.get(v, 1e9):
                coste[v] = ng; padre[v] = n; heapq.heappush(pq, (ng, v))
    return None, expandidos, vistos

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(inicio, meta, h=manhattan, desempate=True):
    # Con desempate guardamos -g (preferimos el nodo mas avanzado ante igual f):
    # asi A* avanza en un haz en vez de expandir todo el abanico de caminos empatados.
    pq = [(h(inicio, meta), 0, inicio)]; padre = {}; coste = {inicio: 0}
    expandidos = 0; vistos = set()
    while pq:
        f, g_key, n = heapq.heappop(pq); g = -g_key if desempate else g_key
        if n in vistos: continue
        vistos.add(n); expandidos += 1
        if n == meta: return reconstruir(padre, meta), expandidos, vistos
        for v in vecinos(n):
            ng = g + 1
            if ng < coste.get(v, 1e9):
                coste[v] = ng; padre[v] = n
                heapq.heappush(pq, (ng + h(v, meta), -ng if desempate else ng, v))
    return None, expandidos, vistos

resultados = {}
for nombre, fn in [("BFS", bfs), ("DFS", dfs), ("UCS", ucs), ("A*", astar)]:
    camino, expandidos, vistos = fn(INICIO, META)
    resultados[nombre] = (camino, expandidos, vistos)
    print(f"{nombre:4}  ->  camino de {len(camino):3} pasos   |   casillas miradas: {expandidos}")


**Lee la tabla.** BFS, UCS y A\* encuentran el mismo camino óptimo (mismo número de pasos), pero
fíjate en *casillas miradas*: A\* mira una fracción de lo que mira BFS, porque su heurística lo empuja
hacia la meta en vez de explorar en todas direcciones (con un buen desempate llega a mirar casi solo
las casillas del propio camino). DFS, en cambio, llega rápido a *algo*, pero su ruta es larguísima:
se mete por un pasillo y no vuelve aunque haya algo mejor. **Rapidez no es lo mismo que calidad.**


## 3. Verlo: qué exploró cada uno

Pintamos en gris las casillas que cada algoritmo tuvo que mirar, y en negro el camino final. La nube
gris de BFS llena medio mapa; la de A\* es un haz dirigido a la meta. Esa es, en una imagen, la
diferencia entre fuerza bruta y criterio.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, nombre in zip(axes, ["BFS", "DFS", "UCS", "A*"]):
    camino, expandidos, vistos = resultados[nombre]
    lienzo = np.ones((*GRID.shape, 3))
    lienzo[GRID == 1] = [0.1, 0.1, 0.1]
    for (f, c) in vistos: lienzo[f, c] = [0.78, 0.78, 0.78]
    for (f, c) in camino: lienzo[f, c] = [0.0, 0.0, 0.0]
    ax.imshow(lienzo); ax.set_title(f"{nombre}: {expandidos} miradas")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 4. El corazón de A\*: heurística admisible y desempate

A\* es bueno **gracias a** la heurística, pero solo si esta cumple una condición: ser **admisible**, es
decir, **nunca sobreestimar** el coste que falta. La distancia de Manhattan lo cumple (en línea recta
por la cuadrícula nunca es más que rodeando), y por eso A\* con Manhattan **garantiza el camino
óptimo**. Si usaras una heurística que sobreestima, A\* correría más pero podría devolverte un camino
peor: perderías la garantía. Esa es la regla de oro de las heurísticas, y la pondremos a prueba con
números más abajo.

Hay un segundo detalle, más sutil, que explica por qué A\* mira tan **poquísimas** casillas: el
**desempate**. En una cuadrícula hay muchísimos caminos óptimos empatados (todos los que avanzan
monótonamente hacia la meta miden lo mismo). Sin un criterio extra, A\* los expande **todos**, llenando
un rectángulo. Si, ante igual $f$, preferimos el nodo **más avanzado** (mayor $g$), A\* avanza en un haz
estrecho. Mismo resultado óptimo, una fracción del trabajo. Lo medimos comparando A\* con y sin ese
desempate.


In [ ]:
print("variante de A*           | casillas miradas | longitud del camino | ¿óptimo?")
optimo = len(resultados["A*"][0])
for nombre, desemp in [("sin desempate", False), ("con desempate (mayor g)", True)]:
    camino, expandidos, _ = astar(INICIO, META, desempate=desemp)
    print(f"  {nombre:<23}|       {expandidos:>4}         |        {len(camino):>3}          |  "
          f"{'sí' if len(camino)==optimo else 'no'}")
print("\nAmbos dan el camino optimo, pero el desempate reduce drasticamente lo que A* explora:")
print("sin desempate expande casi todo el rectangulo de caminos empatados; con el, va en linea.")


**Lo que enseña ese experimento.** Las dos variantes encuentran el mismo camino óptimo, pero el
desempate recorta a una fracción las casillas exploradas. Sin él, A\* malgasta tiempo expandiendo
decenas de caminos que miden exactamente lo mismo; con él, elige uno y avanza. Es un recordatorio de
que en los algoritmos de búsqueda los **detalles de implementación** (qué sacas primero de la frontera)
pueden importar tanto como la idea general.


## 5. Dos heurísticas admisibles: Manhattan y euclídea

Hasta aquí hemos usado una sola heurística. Pero hay varias que «nunca sobreestiman», y entre ellas no
todas valen lo mismo. Compararemos tres formas de estimar lo que falta:

- **Ninguna** ($h=0$): A\* se convierte en UCS y explora a lo bestia.
- **Distancia euclídea** (en línea recta): admisible, porque ir en diagonal por el aire nunca es más
  largo que moverse por la cuadrícula. Pero da estimaciones *más pequeñas* que Manhattan, así que tira
  con menos fuerza hacia la meta.
- **Distancia de Manhattan**: la más ajustada de las admisibles en una cuadrícula con movimientos en
  cruz. Cuanto más se acerca la heurística al coste real (sin pasarse), **menos** casillas mira A\*.

Idea de fondo: entre las heurísticas admisibles, **gana la que más se aproxima por debajo**. Vamos a
medirlo.


In [ ]:
def vecinos_g(p, grid):
    # Como vecinos(), pero sobre un mapa cualquiera (1 = muro).
    f, c = p
    for df, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nf, nc = f+df, c+dc
        if 0 <= nf < grid.shape[0] and 0 <= nc < grid.shape[1] and grid[nf, nc] != 1:
            yield (nf, nc)

def cero(a, b):       return 0.0
def euclidea(a, b):   return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5

def busqueda(inicio, meta, grid, h=manhattan, w_h=1.0, usar_g=True, coste=None, desempate=True):
    # Motor unico que cubre UCS, A*, A* ponderado y busqueda voraz:
    #   prioridad f = (g si usar_g) + w_h * h(n, meta)
    #   coste(casilla): coste de ENTRAR en esa casilla (None = 1 por paso).
    # Devuelve (camino, casillas_miradas, vistos, coste_total_del_camino).
    pq = [(w_h*h(inicio, meta), 0, inicio)]
    padre = {}; g_real = {inicio: 0}; expandidos = 0; vistos = set()
    while pq:
        f, gkey, n = heapq.heappop(pq)
        if n in vistos: continue
        vistos.add(n); expandidos += 1
        if n == meta:
            return reconstruir(padre, meta), expandidos, vistos, g_real[meta]
        for v in vecinos_g(n, grid):
            if v in vistos: continue
            paso = 1 if coste is None else coste(v)
            ng = g_real[n] + paso
            if ng < g_real.get(v, 1e18):
                g_real[v] = ng; padre[v] = n
                base = ng if usar_g else 0.0
                heapq.heappush(pq, (base + w_h*h(v, meta), -ng if desempate else ng, v))
    return None, expandidos, vistos, None

optimo = len(resultados["A*"][0])
print("heuristica (admisible) | casillas miradas | longitud | ¿óptimo?")
for nombre, hfun in [("ninguna (= UCS)", cero), ("euclidea", euclidea), ("Manhattan", manhattan)]:
    cam, exp, _, _ = busqueda(INICIO, META, GRID, h=hfun)
    print(f"  {nombre:<20}|      {exp:>4}        |   {len(cam):>4}   |  {'sí' if len(cam)==optimo else 'no'}")


**Lo que sale.** Las tres dan el **mismo camino óptimo** (todas son admisibles), pero el número de
casillas miradas cae según la heurística aprieta más: sin heurística (UCS) explora muchísimo;
con la euclídea, bastante menos; con Manhattan, lo mínimo de las tres. Moraleja: no basta con que una
heurística sea admisible, **interesa la más ajustada** de las admisibles. La euclídea infraestima un
poco más (mide en diagonal por el aire un movimiento que en realidad es en cruz), y esa flojera se paga
en casillas de más.


## 6. Búsqueda voraz: ir solo hacia donde parece más cerca

Llevamos el péndulo al otro extremo. La **búsqueda voraz** (greedy) ordena la frontera **solo por la
heurística** $h$, ignorando por completo lo ya recorrido ($g$). Dicho en cristiano: en cada paso va
hacia la casilla que *parece* más cercana a la meta, sin tener en cuenta lo que lleva andado. Es como
conducir mirando solo la línea recta al destino, sin pensar si la calle que tomas te obliga después a
un rodeo enorme.

¿El resultado? Suele mirar **poquísimo** (es muy directa), pero **no garantiza el camino más corto**:
basta que la casilla más prometedora la meta en un brazo largo para que se quede con una ruta peor.
Optimismo barato, a veces caro. Lo comparamos con A\*.


In [ ]:
optimo = len(resultados["A*"][0])
print("algoritmo        | casillas miradas | longitud | ¿óptimo?")
for nombre, kw in [("A* (g + h)", dict(usar_g=True)), ("voraz (solo h)", dict(usar_g=False))]:
    cam, exp, _, _ = busqueda(INICIO, META, GRID, h=manhattan, **kw)
    print(f"  {nombre:<14} |      {exp:>4}        |   {len(cam):>4}   |  {'sí' if len(cam)==optimo else 'no'}")
print(f"\n(El óptimo en este mapa son {optimo} pasos.)")
print("La voraz es muy directa, pero su garantia de optimalidad es ninguna: depende del mapa.")


**Lo que ves aquí.** En esta explanada tan despejada la voraz tiene suerte y casi no hay trampa: el
edificio se rodea igual de bien yendo a ojo. Pero esa suerte **es del mapa, no del algoritmo**. En
cuanto el terreno tenga pasillos y callejones, la avaricia se paga: lo comprobarás en el laberinto de
la siguiente sección, donde la voraz se traga un rodeo que A\* esquiva. Recuerda la regla: la voraz
optimiza «lo que parece», no «lo que cuesta».


## 7. Otro mapa: un laberinto de pasillos (donde DFS hace el ridículo)

La explanada era benévola con casi todos. Un **laberinto** —obstáculos repartidos que dejan pasillos
estrechos— separa el grano de la paja. Aquí:

- **DFS** se mete por un pasillo, tira hasta el fondo y, cuando rebota, ya ha dado un rodeo descomunal:
  su camino sale larguísimo aunque encuentre la salida.
- **BFS** y **A\*** siguen dando el camino más corto; A\* además mirando mucho menos que BFS.
- La **voraz** suele tropezar: persigue la dirección a la meta y se come algún callejón.

Generamos un laberinto con obstáculos al azar (semilla fija, para que te salga igual) y comparamos.


In [ ]:
def bfs_g(inicio, meta, grid):
    frontera = deque([inicio]); padre = {}; vistos = {inicio}; exp = 0
    while frontera:
        n = frontera.popleft(); exp += 1
        if n == meta: return reconstruir(padre, meta), exp, vistos
        for v in vecinos_g(n, grid):
            if v not in vistos:
                vistos.add(v); padre[v] = n; frontera.append(v)
    return None, exp, vistos

def dfs_g(inicio, meta, grid):
    pila = [inicio]; padre = {}; vistos = {inicio}; exp = 0
    while pila:
        n = pila.pop(); exp += 1
        if n == meta: return reconstruir(padre, meta), exp, vistos
        for v in vecinos_g(n, grid):
            if v not in vistos:
                vistos.add(v); padre[v] = n; pila.append(v)
    return None, exp, vistos

rng = np.random.default_rng(3)
LAB = (rng.random((23, 35)) < 0.28).astype(int)   # 28% de casillas son muro
SL, ML = (1, 1), (21, 33)
for (f, c) in (SL, ML):                            # despejamos inicio y meta y su entorno
    LAB[max(0,f-1):f+2, max(0,c-1):c+2] = 0

lab_res = {}
lab_res["BFS"]   = bfs_g(SL, ML, LAB)
lab_res["DFS"]   = dfs_g(SL, ML, LAB)
cam, exp, vis, _ = busqueda(SL, ML, LAB, h=manhattan);                lab_res["A*"]    = (cam, exp, vis)
cam, exp, vis, _ = busqueda(SL, ML, LAB, h=manhattan, usar_g=False);  lab_res["voraz"] = (cam, exp, vis)

opt_lab = len(lab_res["BFS"][0])
print("algoritmo | longitud del camino | casillas miradas | ¿óptimo?")
for nombre in ["BFS", "DFS", "A*", "voraz"]:
    cam, exp, _ = lab_res[nombre]
    print(f"  {nombre:<7} |        {len(cam):>4}         |       {exp:>4}        |  {'sí' if len(cam)==opt_lab else 'no'}")
print(f"\nEl camino mas corto del laberinto son {opt_lab} pasos.")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.0))
for ax, nombre in zip(axes, ["BFS", "DFS", "A*", "voraz"]):
    cam, exp, vis = lab_res[nombre]
    lienzo = np.ones((*LAB.shape, 3))
    lienzo[LAB == 1] = [0.12, 0.12, 0.12]
    for (f, c) in vis: lienzo[f, c] = [0.80, 0.80, 0.80]
    for (f, c) in cam: lienzo[f, c] = [0.0, 0.0, 0.0]
    ax.imshow(lienzo); ax.set_title(f"{nombre}: {len(cam)} pasos, {exp} miradas")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


**La imagen lo grita.** El camino negro de DFS serpentea por medio laberinto: encuentra la salida,
pero su ruta no se parece a la más corta. BFS y A\* trazan caminos del mismo largo (el mínimo), solo
que la mancha gris de A\* —lo que tuvo que mirar— es bastante más pequeña que la de BFS. Y la voraz, si
en la tabla salió peor que el óptimo, te lo enseña aquí en negro: un rodeo por haber perseguido la
dirección equivocada. Mismo mapa, cuatro maneras de recorrerlo, garantías muy distintas.


## 8. A\* ponderado: la palanca entre velocidad y optimalidad

En la explanada del principio todos los algoritmos lo tenían fácil; en el laberinto empiezan a
distinguirse. Es el sitio ideal para una última herramienta. ¿Y si tuvieras prisa y te conformaras con
un camino *casi* óptimo a cambio de mirar mucho menos? Esa palanca existe y se llama **A\* ponderado**:
ordena la frontera por $f = g + w\cdot h$ con un peso $w>1$. Al multiplicar la heurística la haces
**más agresiva**: A\* se lanza hacia la meta con menos miramientos. El precio: con $w>1$ la heurística
**ya no es admisible** (ahora puede sobreestimar), así que **pierdes la garantía** de optimalidad.

Es la misma idea que una «heurística inadmisible»: multiplicar Manhattan por 1.5 es exactamente
$w=1.5$. Barremos varios pesos sobre el laberinto y miramos el compromiso: casillas miradas frente a
longitud del camino.


In [ ]:
print("  w   | casillas miradas | longitud | exceso sobre el óptimo")
for w in [1.0, 1.2, 1.5, 2.0, 5.0]:
    cam, exp, _, _ = busqueda(SL, ML, LAB, h=manhattan, w_h=w)
    exceso = 100.0 * (len(cam) - opt_lab) / opt_lab
    if w == 1.0:           marca = "  (admisible, óptimo)"
    elif len(cam) > opt_lab: marca = "  (inadmisible: pierde el óptimo)"
    else:                  marca = "  (inadmisible, aquí no se nota)"
    print(f"  {w:>3} |      {exp:>4}        |   {len(cam):>4}   |   +{exceso:4.1f}%{marca}")
print(f"\nEl óptimo del laberinto son {opt_lab} pasos. Al subir w miras MUCHO menos, pero el camino")
print("se alarga por encima del óptimo: velocidad a cambio de optimalidad, la palanca en tus manos.")


**Cómo leerlo.** En $w=1$ tienes el A\* de siempre: óptimo, pero mirando bastante. En cuanto subes
$w$, la columna de *casillas miradas* se desploma (vas mucho más directo) mientras la *longitud* sube
un poco por encima del óptimo: ese «exceso» es, literalmente, lo que pagas por correr más. No es un
fallo, es una **decisión de ingeniería**: un videojuego que mueve cien personajes a la vez prefiere
rutas un pelín peores calculadas al instante; un GPS que planifica una sola ruta prefiere la óptima
aunque tarde un poco más. Y fíjate en el caso extremo: con $w$ grande, A\* ponderado se parece cada vez
más a la búsqueda voraz (que es, en el fondo, $w=\infty$: solo heurística).


## 9. Terreno con coste: barro que cuesta más (y BFS deja de ser óptimo)

Hasta ahora **cada paso costaba 1**, así que «más corto» y «más barato» eran lo mismo. El mundo real no
es así: cruzar un cruce con semáforo, una cuesta o un barrizal cuesta más que avanzar por terreno
llano. Ponemos una **franja de barro** que cuesta varias veces más, con una zona despejada por la que
se puede rodear.

Y aquí salta la diferencia clave:

- **BFS** solo cuenta **pasos**: te clava la ruta de menos casillas... aunque cruce el barro de lleno.
- **UCS** y **A\*** cuentan **coste**: prefieren dar un rodeo por terreno barato si sale más económico.

Para que A\* siga siendo óptimo con costes, su heurística debe seguir sin sobreestimar: como ninguna
casilla cuesta menos de 1, Manhattan sigue siendo admisible. Medimos pasos, coste y miradas.


In [ ]:
COSTE = np.ones((15, 25))         # terreno llano: cada casilla cuesta 1 al entrar
COSTE[0:12, 10:13] = 9            # franja de barro (cuesta 9), abierta por abajo (filas 12-14)
GRID_T = np.zeros((15, 25), int)  # sin muros: aqui el obstaculo es el coste, no el bloqueo
ST, MT = (7, 1), (7, 23)
coste_de = lambda v: COSTE[v[0], v[1]]
def coste_camino(cam):
    return sum(coste_de(c) for c in cam[1:])   # el inicio no se "entra"

# BFS ignora el coste (cuenta pasos); UCS y A* lo tienen en cuenta.
cam_bfs, exp_bfs, _      = bfs_g(ST, MT, GRID_T)
cam_ucs, exp_ucs, _, c_ucs = busqueda(ST, MT, GRID_T, h=cero,      coste=coste_de)   # UCS
cam_ast, exp_ast, _, c_ast = busqueda(ST, MT, GRID_T, h=manhattan, coste=coste_de)   # A*

print("algoritmo | pasos | coste real del camino | casillas miradas")
for nombre, cam, exp in [("BFS", cam_bfs, exp_bfs), ("UCS", cam_ucs, exp_ucs), ("A*", cam_ast, exp_ast)]:
    print(f"  {nombre:<5}   |  {len(cam)-1:>3}  |          {coste_camino(cam):>4.0f}         |       {exp:>4}")
print("\nBFS hace menos pasos pero cruza el barro: paga mas. UCS y A* rodean y salen mas baratos.")
print("UCS y A* dan el MISMO coste optimo; A* lo consigue mirando menos casillas.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (nombre, cam) in zip(axes, [("BFS: cuenta pasos", cam_bfs), ("A*: cuenta coste", cam_ast)]):
    fondo = 1.0 - 0.06 * COSTE          # el barro se ve mas oscuro
    lienzo = np.stack([fondo]*3, axis=-1)
    for (f, c) in cam: lienzo[f, c] = [0.85, 0.1, 0.1]   # camino en rojo
    ax.imshow(np.clip(lienzo, 0, 1))
    ax.set_title(f"{nombre}  (coste {coste_camino(cam):.0f})")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


**Lo que demuestra.** BFS clava la ruta de menos pasos y, como en la imagen, la mete recta por el
barro: pocos pasos, coste alto. UCS y A\* prefieren rodear por la zona despejada: algún paso más, pero
**coste total menor**, y exactamente el mismo entre los dos (UCS y A\* dan el óptimo en coste). La
diferencia entre ellos está en el esfuerzo: A\* llega a ese óptimo mirando bastantes menos casillas
gracias a la heurística. La lección: **cuando los pasos no cuestan lo mismo, «más corto» y «más barato»
se separan**, y solo los algoritmos que miran el coste (UCS, A\*) te dan lo segundo.


## 10. Pruébalo tú

1. **Mueve la meta** cerca del inicio (`META = (16, 5)`) y repite la sección 2. ¿Se acortan las
   diferencias entre BFS y A\*? La heurística luce más cuanto más lejos está el objetivo.
2. **Sube el peso** del A\* ponderado a `w = 20` en la sección 8. ¿Hasta dónde se alarga el camino?
   ¿Llega a parecerse a la búsqueda voraz (que es, en el fondo, $w=\infty$)?
3. **Engorda el barro:** cambia el `9` de la franja por `2` en la sección 9. ¿Sigue compensando rodear,
   o a BFS ya casi le da igual? Hay un punto en el que cruzar deja de ser caro.
4. **Cambia la semilla** del laberinto (`default_rng(3)` por otro número) y vuelve a mirar a la voraz y
   a DFS. ¿Cómo de mal lo pasan según el mapa? Si la meta queda incomunicada, sáltala y prueba otra.
5. **Heurística inadmisible a propósito:** prueba la euclídea multiplicada por 3 en la sección 5. Mirará
   menos, pero ¿sigue saliendo el camino óptimo? Comprueba que la admisibilidad no es un capricho.


## 11. Errores comunes

- **Creer que DFS es malo siempre.** DFS usa poquísima memoria y es ideal cuando cualquier solución
  vale o el árbol es muy profundo; lo que no garantiza es la *más corta*, como viste en el laberinto.
- **Pensar que la voraz y A\* son lo mismo.** La voraz mira solo $h$ (lo que falta); A\* mira $g+h$ (lo
  andado más lo que falta). Por eso la voraz es rápida pero se equivoca, y A\* no.
- **Usar una heurística inadmisible sin saberlo** y extrañarse de que A\* no dé el óptimo. Multiplicar
  la heurística (A\* ponderado) corre más, pero pierde la garantía: es un trato, no gratis.
- **Olvidar el desempate.** Sin él, A\* expande todo el «rectángulo» de caminos empatados; con un buen
  desempate, avanza en un haz. Lo viste en los números.
- **Confundir coste y pasos.** Con costes distintos por casilla, «más corto» y «más barato» dejan de
  coincidir: BFS te da lo primero, UCS y A\* lo segundo. Elige el algoritmo según lo que de verdad
  quieras minimizar.


## 12. Qué te llevas

- Buscar es explorar un **espacio de estados**; todos estos algoritmos hacen eso, lo que cambia es *en
  qué orden* miran, y eso lo decide su estructura de datos (cola, pila, cola de prioridad).
- Cada uno ofrece **garantías** distintas: BFS/UCS/A\* dan el óptimo, DFS no; la voraz tampoco. A\* da
  el óptimo *y* mira mucho menos, **si** la heurística es admisible.
- Una **heurística** que estima «cuánto falta» convierte una búsqueda ciega en una dirigida. Entre las
  admisibles, gana la más ajustada (Manhattan por encima de la euclídea aquí).
- La palanca **velocidad frente a optimalidad** (A\* ponderado, búsqueda voraz) es una decisión de
  ingeniería, no un fallo: pagas camino a cambio de mirar menos.
- Cuando los pasos **cuestan distinto** (barro, cuestas, semáforos), solo los algoritmos que miran el
  coste (UCS, A\*) te dan la ruta más barata. BFS solo sabe de pasos.

**Para seguir:** el capítulo 4 conecta esto con agentes modernos (de algoritmo a *política*); los
capítulos 5-7, con restricciones (CSP); y verás A\* reaparecer cada vez que algo tenga que planificar
pasos hacia un objetivo, incluido un agente LLM.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*